### Chroma Vector Store (langchain_chroma)

Chroma is an open-source, developer-first vector database designed for embedding storage, metadata search, and retrieval.

It supports both in-memory execution and local disk persistence.

### Complete CRUD Operations Lifecycle

1. **Create**: Building database from documents (Chroma.from_documents) and inserting additional items (db.add_documents).
2. **Read / Search**: Executing similarity_search, similarity_search_with_score, max_marginal_relevance_search (MMR), and metadata filtering (filter=...).
3. **Update**: Modifying document content or metadata by ID (db.update_document).
4. **Delete**: Removing target document vectors by ID (db.delete(ids=[...])).

In [1]:
import os
import tempfile
from dotenv import load_dotenv, find_dotenv
from langchain_chroma import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings
# from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

load_dotenv(find_dotenv())

# Use isolated temp directory to prevent SQLite read-only lock errors on re-execution
persist_dir = tempfile.mkdtemp(prefix="chroma_crud_")

# Initialize Google Generative AI embeddings (commented out HuggingFaceEmbeddings)
# embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

# Initial dataset
initial_docs = [
    Document(page_content="Python is a versatile programming language for AI and backend development.", metadata={"category": "tech", "doc_id": "1"}),
    Document(page_content="LangChain provides abstractions for building RAG applications.", metadata={"category": "tech", "doc_id": "2"}),
    Document(page_content="Neapolitan pizza is baked at high temperatures in wood-fired ovens.", metadata={"category": "cooking", "doc_id": "3"})
]

# --- CREATE ---
db = Chroma.from_documents(
    documents=initial_docs,
    embedding=embeddings,
    collection_name="tech_and_food",
    persist_directory=persist_dir
)
print("=== 1. CREATE ===")
print(f"Created Chroma database with {len(initial_docs)} documents.")

# Add new document with explicit ID
new_doc = Document(
    page_content="FastAPI allows creating high-performance Python web APIs with type hints.",
    metadata={"category": "tech", "doc_id": "4"}
)
added_ids = db.add_documents([new_doc], ids=["doc_4"])
print(f"Added document with ID: {added_ids}")

# --- READ / SEARCH ---
print("\n=== 2. READ / RETRIEVAL ===")
# Similarity Search
results = db.similarity_search("What framework helps build LLM apps?", k=1)
print(f"Similarity Search Result: {results[0].page_content}")

# Similarity Search with Score
results_with_score = db.similarity_search_with_score("Python programming", k=1)
print(f"Similarity Score Result: {results_with_score[0][0].page_content} (Score: {results_with_score[0][1]:.4f})")

# Metadata Filtered Search
filtered_results = db.similarity_search("high temperature", filter={"category": "cooking"}, k=1)
print(f"Filtered Search (category=cooking): {filtered_results[0].page_content}")

# MMR (Maximal Marginal Relevance) Search
mmr_results = db.max_marginal_relevance_search("Python software development", k=2, fetch_k=4)
print(f"MMR Search Result Count: {len(mmr_results)}")

# --- UPDATE ---
print("\n=== 3. UPDATE ===")
updated_doc = Document(
    page_content="FastAPI is an asynchronous Python web framework built on Starlette and Pydantic.",
    metadata={"category": "tech", "doc_id": "4", "updated": True}
)
db.update_document(document_id="doc_4", document=updated_doc)
updated_search = db.similarity_search("FastAPI Starlette", k=1)
print(f"Updated Document Content: {updated_search[0].page_content}")

# --- DELETE ---
print("\n=== 4. DELETE ===")
db.delete(ids=["doc_4"])
post_delete_search = db.similarity_search("FastAPI Starlette", k=1)
print(f"Top Result After Deleting doc_4: {post_delete_search[0].page_content}")


=== 1. CREATE ===
Created Chroma database with 3 documents.
Added document with ID: ['doc_4']

=== 2. READ / RETRIEVAL ===
Similarity Search Result: LangChain provides abstractions for building RAG applications.
Similarity Score Result: Python is a versatile programming language for AI and backend development. (Score: 0.5471)
Filtered Search (category=cooking): Neapolitan pizza is baked at high temperatures in wood-fired ovens.
MMR Search Result Count: 2

=== 3. UPDATE ===
Updated Document Content: FastAPI is an asynchronous Python web framework built on Starlette and Pydantic.

=== 4. DELETE ===
Top Result After Deleting doc_4: Python is a versatile programming language for AI and backend development.
